# 4b. Post-Shift Model Fitting

**Purpose**: Fit the best-fit model's parameters to post-shift sessions
and track how they evolve. Does η spike and decay? Does σ_percep change?

**Depends on**: 3a/3c (model assignment per animal), 4a (shift detection),
2a Section 8 (GP-linked recovery validation).

**Protocol**:
1. Load model assignment from 3c consensus
2. For each shifted animal, using its assigned model:
   a. Fit static parameters per session (sliding window of 3 sessions)
   b. Fit GP-linked trajectory (η varies, others constant)
3. Plot parameter trajectories aligned to shift
4. Compare to SLDS states from notebook 5

In [1]:
import sys, os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import pickle
import warnings
warnings.filterwarnings('ignore')

from behav_utils.data.loading import load_experiment
from behav_utils.data.selection import select_sessions, fitting_data_from_sessions
from behav_utils.plotting.styles import COLOURS, apply_style

from analysis.adaptation import detect_first_manipulation, adaptation_trajectory
from analysis.grid_search import grid_search_cv, DEFAULT_GRID, COARSE_GRID

apply_style()
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

In [2]:
CONFIG_PATH = Path('../config.yaml')
STAGE = 'Full_Task_Cont'
CONSENSUS_PATH = Path('../results/model_assignments.csv')

# Sliding window for per-session fitting
WINDOW_SIZE = 3      # sessions pooled per window
WINDOW_STRIDE = 1    # sessions to advance per step

## 1. Load Data & Model Assignments

In [3]:
experiment = load_experiment(CONFIG_PATH)

# Load model assignments from 3c
if CONSENSUS_PATH.exists():
    assignments = pd.read_csv(CONSENSUS_PATH)
    print(f'Loaded assignments for {len(assignments)} animals')
    print(assignments[['animal_id', 'final_model', 'confidence']].to_string(index=False))
else:
    print(f'No consensus file at {CONSENSUS_PATH}')
    print('TODO: run notebook 3c first, or assign manually below')
    # Manual fallback:
    assignments = pd.DataFrame({
        'animal_id': experiment.animal_ids,
        'final_model': ['BE'] * len(experiment.animal_ids),
        'confidence': ['low'] * len(experiment.animal_ids),
    })

Loaded 12 animals, 433 total sessions
No consensus file at ../results/model_assignments.csv
TODO: run notebook 3c first, or assign manually below


In [4]:
# Find shifted animals
shifted = {}
for aid in experiment.animal_ids:
    animal = experiment.get_animal(aid)
    manip = detect_first_manipulation(animal, stage=STAGE)
    if manip:
        all_sess = animal.get_sessions(stage=STAGE)
        shift_idx = manip['session_idx']
        baseline = all_sess[:shift_idx]
        post = all_sess[shift_idx:]
        model = assignments.loc[
            assignments['animal_id'] == aid, 'final_model'
        ].values
        model = model[0] if len(model) > 0 else 'BE'

        shifted[aid] = {
            'animal': animal,
            'baseline': baseline,
            'post': post,
            'shift': manip,
            'model': model,
        }
        print(f"{aid}: {model}, shift at session {manip['global_session_idx']}, "
              f"{len(post)} post-shift sessions")

print(f'\n{len(shifted)} animals with shifts')

SS01: BE, shift at session 44, 7 post-shift sessions
SS04: BE, shift at session 56, 7 post-shift sessions
SS05: BE, shift at session 34, 7 post-shift sessions
SS06: BE, shift at session 34, 7 post-shift sessions
SS07: BE, shift at session 34, 7 post-shift sessions
SS08: BE, shift at session 34, 7 post-shift sessions
SS09: BE, shift at session 25, 7 post-shift sessions
SS11: BE, shift at session 28, 4 post-shift sessions
SS13: BE, shift at session 27, 5 post-shift sessions

9 animals with shifts


## 2. Sliding-Window Grid-Search Fitting

For each window of WINDOW_SIZE sessions, find the best-fit parameters
using grid-search on the update matrix. This gives a coarse parameter
trajectory without needing SBI.

In [5]:
window_results = {}  # aid -> list of {session_centre, params, error}

for aid, info in shifted.items():
    model_type = info['model']
    all_sess = info['baseline'] + info['post']
    n_baseline = len(info['baseline'])

    results = []
    grid = COARSE_GRID[model_type]  # use coarse for speed

    for start in range(0, len(all_sess) - WINDOW_SIZE + 1, WINDOW_STRIDE):
        window = all_sess[start:start + WINDOW_SIZE]
        centre = start + WINDOW_SIZE // 2
        relative = centre - n_baseline  # negative = baseline

        try:
            cv = grid_search_cv(
                window, model_type, grid,
                n_folds=2, seed=42, burn_in=1000, n_jobs=-1,
            )
            results.append({
                'session_centre': centre,
                'relative_session': relative,
                'params': cv['best_params_single'],
                'error': cv['avg_test_error'],
            })
        except Exception as e:
            print(f'  {aid} window {start}: {e}')

    window_results[aid] = results
    print(f"{aid}: {len(results)} windows fitted")

SS01: 48 windows fitted


KeyboardInterrupt: 

## 3. Parameter Trajectories

In [ ]:
for aid, results in window_results.items():
    if not results:
        continue

    model_type = shifted[aid]['model']
    df = pd.DataFrame(results)

    # Extract parameter names from first result
    param_names = [k for k in df['params'].iloc[0].keys()]

    n_params = len(param_names)
    fig, axes = plt.subplots(1, n_params + 1, figsize=(4 * (n_params + 1), 3.5))

    # Plot each parameter
    for j, pname in enumerate(param_names):
        ax = axes[j]
        vals = [r['params'][pname] for r in results]
        rel = [r['relative_session'] for r in results]

        bl_mask = np.array(rel) < 0
        post_mask = np.array(rel) >= 0

        ax.plot(np.array(rel)[bl_mask], np.array(vals)[bl_mask],
                'o-', ms=4, color='steelblue')
        ax.plot(np.array(rel)[post_mask], np.array(vals)[post_mask],
                'o-', ms=4, color='darkorange')
        ax.axvline(0, color='black', ls=':', lw=0.8)
        ax.set_title(pname, fontsize=10)
        ax.set_xlabel('Sessions from shift')

    # Plot fit error
    ax = axes[-1]
    errors = [r['error'] for r in results]
    rel = [r['relative_session'] for r in results]
    ax.plot(rel, errors, 'o-', ms=4, color='grey')
    ax.axvline(0, color='black', ls=':', lw=0.8)
    ax.set_title('Fit error (MSE)', fontsize=10)
    ax.set_xlabel('Sessions from shift')

    shift_desc = shifted[aid]['shift']['details']
    fig.suptitle(f"{aid} ({model_type}): {shift_desc.get('before','?')} → "
                 f"{shift_desc.get('after','?')}",
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 4. GP-Linked SBI Trajectory

More principled than sliding-window grid search: fit all sessions
jointly with a GP prior on η, enforcing smoothness.

**TODO**: This requires the SBIFitter to be fully operational.
The infrastructure is set up in inference/sbi_fitter.py;
validation is in 2a Section 8.

In [ ]:
# Placeholder — fill in when SBIFitter is validated
print('GP-linked SBI trajectory fitting: TODO')
print('Run 2a Section 8 first to validate the pipeline')
print('Then use SBIFitter with GPLink on eta_learning/gamma')

## 5. Connect to SLDS States

Do parameter changes align with SLDS state transitions from notebook 5?

In [ ]:
slds_path = Path('../results/slds/slds_results.pkl')
if slds_path.exists():
    with open(slds_path, 'rb') as f:
        slds_results = pickle.load(f)

    hmm_labels = slds_results.get('hmm_labels', {})
    state_names = slds_results.get('hmm_state_names', {})

    for aid, results in window_results.items():
        if aid not in hmm_labels or not results:
            continue

        labels = hmm_labels[aid]
        # Align: window centres to session indices
        animal = shifted[aid]['animal']
        all_sess = animal.get_sessions(stage=STAGE)

        # Plot param trajectory with state labels
        model_type = shifted[aid]['model']
        lr_param = 'eta_learning' if model_type == 'BE' else 'gamma'

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

        # Top: learning rate
        rel = [r['relative_session'] for r in results]
        lr_vals = [r['params'].get(lr_param, np.nan) for r in results]
        ax1.plot(rel, lr_vals, 'o-', ms=4, color='steelblue')
        ax1.axvline(0, color='black', ls=':', lw=0.8)
        ax1.set_ylabel(lr_param)
        ax1.set_title(f'{aid}: {lr_param} trajectory + SLDS states')

        # Bottom: SLDS states
        n_baseline = len(shifted[aid]['baseline'])
        sess_relative = np.arange(len(labels)) - n_baseline
        state_colours = plt.cm.Set2(np.linspace(0, 1, max(labels) + 1))

        for k in range(max(labels) + 1):
            mask = labels == k
            ax2.scatter(sess_relative[mask],
                        np.ones(mask.sum()) * 0.5,
                        c=[state_colours[k]], s=60, marker='s',
                        label=state_names.get(k, f'S{k}'))
        ax2.axvline(0, color='black', ls=':', lw=0.8)
        ax2.set_yticks([])
        ax2.set_xlabel('Sessions from shift')
        ax2.legend(fontsize=7)

        plt.tight_layout()
        plt.show()
else:
    print(f'No SLDS results at {slds_path} — run notebook 5 first')

## 6. Save Results

In [ ]:
# results_dir = Path('../results/post_shift_fitting')
# results_dir.mkdir(parents=True, exist_ok=True)

# save_data = {
#     'window_results': window_results,
#     'window_size': WINDOW_SIZE,
#     'window_stride': WINDOW_STRIDE,
# }

# with open(results_dir / 'window_fitting_results.pkl', 'wb') as f:
#     pickle.dump(save_data, f)
# print(f'Saved to {results_dir}')

## Interpretation

**If η spikes at shift and decays:** Consistent with the model adequacy
framework — the animal increases its learning rate when the model is
inadequate, then reduces it as the new model becomes adequate.

**If σ_percep changes:** Suggests perceptual adaptation beyond just
model updating — the animal is changing how it processes stimuli.

**If fit error spikes at shift:** The model itself may be inadequate
during rapid adaptation (the stationary-parameters assumption breaks).
This motivates the GP-linked approach.

**If SLDS transitions align with parameter changes:** Model-free (SLDS)
and model-based (parameter trajectories) characterisations converge.
This strengthens both approaches.